# Study 824 — Cochrane-Piazzesi Factor — the teardown

The HAC predictive regression, the tent loadings, the size-distorted *t*, the Campbell-Thompson out-of-sample R², the 1,000-draw block placebo, the two-era cut, the costed duration timer, and the 20-seed synthetic control.

In [1]:
R = {'start': '2002-01-02', 'end': '2026-06-30', 'n_rows': 6162, 'fingerprint': '03a5e9844a31', 'n': 5760, 'nw_lags': 378, 'r2': 0.2261, 'cp_slope': 1.0, 'cp_slope_t': 2.618, 'avg_rx_bps': 166.7, 'load_const': -0.0696, 'load_yshort': -0.0587, 'load_f1': -2.1058, 'load_f2': 4.9245, 'load_f3': -0.9744, 'load_t_const': -0.91, 'load_t_yshort': -0.06, 'load_t_f1': -1.12, 'load_t_f2': 0.95, 'load_t_f3': -0.16, 'oos_r2': -0.2722, 'oos_npreds': 227, 'placebo_obs': 0.2261, 'placebo_mean': 0.1795, 'placebo_sd': 0.0541, 'placebo_p': 0.208, 'era1_n': 2620, 'era1_r2': 0.4118, 'era1_t': 3.902, 'era2_n': 2888, 'era2_r2': 0.1139, 'era2_t': 1.717, 'timer2_sharpe': 0.016, 'timer5_sharpe': 0.005, 'bh_sharpe': 0.216, 'switches': 3.7, 'invested': 0.47, 'null_r2_mean': 0.0007, 'null_r2_max': 0.0019, 'null_t_mean': 2.08, 'null_fire': 11, 'null_oos': 0.0186, 'plant_r2': 0.6613, 'plant_t': 100.09, 'plant_oos': 0.6309}

## The headline — regress avg 1y excess return on the forward vector

`avg_rx_{t+252} = γ'·[1, y_short, f_1, f_2, f_3] + e`; the fitted value is the CP factor. HAC lags = 378 (≈ 1.5× the 252-day overlap).

In [2]:
print(f"n = {R['n']}   in-sample R2 = {R['r2']:.4f}   avg excess = {R['avg_rx_bps']:+.1f} bps")
print(f"single-factor predictive slope = {R['cp_slope']:.3f}  NW t = {R['cp_slope_t']:+.3f}")
print('tent loadings (NW t):')
for nm,b,t in [('y_short',R['load_yshort'],R['load_t_yshort']),
               ('f_1',R['load_f1'],R['load_t_f1']),
               ('f_2',R['load_f2'],R['load_t_f2']),
               ('f_3',R['load_f3'],R['load_t_f3'])]:
    print(f'   {nm:>8}: {b:+.3f}  (t={t:+.2f})')
print('  -> peak on f_2 (5->10y forward): the tent. But every single t is insignificant.')

n = 5760   in-sample R2 = 0.2261   avg excess = +166.7 bps
single-factor predictive slope = 1.000  NW t = +2.618
tent loadings (NW t):
    y_short: -0.059  (t=-0.06)
        f_1: -2.106  (t=-1.12)
        f_2: +4.925  (t=+0.95)
        f_3: -0.974  (t=-0.16)
  -> peak on f_2 (5->10y forward): the tent. But every single t is insignificant.


## The naive HAC *t* is size-distorted — the placebo proves it

In [3]:
print(f"block placebo: observed R2 {R['placebo_obs']:.4f} vs null mean {R['placebo_mean']:.4f} "
      f"(sd {R['placebo_sd']:.4f}) -> p = {R['placebo_p']:.4f}")
print('  persistent regressors ALONE manufacture R2 ~ 0.18 -> observed is ~0.9 sigma up, p=0.21')

block placebo: observed R2 0.2261 vs null mean 0.1795 (sd 0.0541) -> p = 0.2080
  persistent regressors ALONE manufacture R2 ~ 0.18 -> observed is ~0.9 sigma up, p=0.21


## Out-of-sample — does it beat the prevailing mean?

In [4]:
print(f"Campbell-Thompson OOS R2 = {R['oos_r2']:+.4f}  ({R['oos_npreds']} forecasts)")
print('  negative -> the CP forecast is WORSE than a constant out of sample')

Campbell-Thompson OOS R2 = -0.2722  (227 forecasts)
  negative -> the CP forecast is WORSE than a constant out of sample


## Robustness — two eras (split 2014-01-01)

In [5]:
print(f"2002-2013 (n={R['era1_n']}): R2={R['era1_r2']:.4f}  NW t={R['era1_t']:+.3f}")
print(f"2014-2026 (n={R['era2_n']}): R2={R['era2_r2']:.4f}  NW t={R['era2_t']:+.3f}")
print('  whatever fit exists is concentrated in the first (QE / falling-rate) half')

2002-2013 (n=2620): R2=0.4118  NW t=+3.902
2014-2026 (n=2888): R2=0.1139  NW t=+1.717
  whatever fit exists is concentrated in the first (QE / falling-rate) half


## The timer — can you get paid for it?

Own TLT when the *out-of-sample* CP forecast is above its rolling median, else cash; one-way cost × traded NAV per switch.

In [6]:
for tag,s in [('2 bps',R['timer2_sharpe']),('5 bps',R['timer5_sharpe'])]:
    print(f"{tag:>5} one-way: net Sharpe {s:+.3f}  vs buy-and-hold TLT {R['bh_sharpe']:+.3f}")
print(f"  ({R['switches']:.1f} switches/yr, invested {R['invested']:.0%}) -> the signal subtracts value")

2 bps one-way: net Sharpe +0.016  vs buy-and-hold TLT +0.216
5 bps one-way: net Sharpe +0.005  vs buy-and-hold TLT +0.216
  (3.7 switches/yr, invested 47%) -> the signal subtracts value


## Synthetic positive control — the machinery is unbiased

Live: R² and OOS R² must be silent on the null and light up on a planted edge. (The raw single-factor HAC *t* is NOT a valid detector here — it fires on the null.)

In [7]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from cp_factor import data, strategy as st
null_r2 = np.array([st.synthetic_detect(data.synthetic_daily(edge=0.0, seed=824+s, n_days=3000))['r2'] for s in range(8)])
null_t  = np.array([st.synthetic_detect(data.synthetic_daily(edge=0.0, seed=824+s, n_days=3000))['cp_slope_t'] for s in range(8)])
plant = st.synthetic_detect(data.synthetic_daily(edge=0.05, seed=824, n_days=3000))
print(f"null (edge=0), 8 seeds: R2 mean {null_r2.mean():.4f} (max {null_r2.max():.4f}) -> SILENT")
print(f"  but raw HAC t mean {null_t.mean():+.2f}, fires |t|>=2 on {(abs(null_t)>=2).sum()}/8 -> size-distorted")
print(f"planted (edge=0.05): R2 = {plant['r2']:.4f} -> the machinery recovers a real edge")

null (edge=0), 8 seeds: R2 mean 0.0011 (max 0.0023) -> SILENT
  but raw HAC t mean +2.22, fires |t|>=2 on 4/8 -> size-distorted
planted (edge=0.05): R2 = 0.6769 -> the machinery recovers a real edge


## Verdict

- **Signal — Weak.** The CP regression shows the right **tent** (peak on the 5→10y forward) and a fat in-sample **R² = 0.226**, but it is **≈0.9σ from a spurious persistent-regressor fit** (placebo p = 0.21; null R² ≈ 0.18), goes **negative out of sample** (OOS R² = -0.27), is confined to the first era (*t* = +3.90 vs +1.72), and its headline HAC *t* = +2.62 lives inside the size-distorted null band (mean +2.08, fires 11/20). The synthetic control's R² is clean (null 0.0007, planted 0.661), so this is the honest Bauer-Hamilton reading, not a bug. Coarse-grid proxy caveat on the Signal axis.
- **Tradability — Mirage.** The CP-timed duration book earns a net Sharpe of ~0.02 (2 bps) / 0.005 (5 bps) versus 0.22 for simply holding TLT — the signal subtracts value. No paycheck.